# Tạo lại các biểu đồ (CSIC) từ mô hình đã train

Notebook này **không train lại** — nó chỉ nạp các model `*_final.pkl` đã có và các
file kết quả JSON để **vẽ lại đúng các biểu đồ** trong `analysis/charts/`
(trừ `flow.png`, và không bao gồm HttpParams/ECML).

Thứ tự:
1. Feature importance (Random Forest) — `feature_importance.png`
2. Tier & Hybrid comparison — `model_comparison.png`
3. Comprehensive comparison (4 ảnh) — `comprehensive_metrics.png`, `precision_vs_recall.png`, `f1_ranking.png`, `radar_comparison.png`
4. Confusion matrices — `confusion_matrices.png`

> Chạy notebook từ trong thư mục gốc dự án (nơi có `apache_log.py`).


## 0. Thiết lập (paths + import)

In [ ]:
import os, sys, json, joblib
import numpy as np
import matplotlib.pyplot as plt

# Tự tìm thư mục gốc (chứa apache_log.py)
ROOT = os.path.abspath('.')
while not os.path.exists(os.path.join(ROOT, 'apache_log.py')) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)

MODELS   = os.path.join(ROOT, 'trained_models')
ANALYSIS = os.path.join(ROOT, 'analysis')
CHARTS   = os.path.join(ANALYSIS, 'charts')
os.makedirs(CHARTS, exist_ok=True)

sys.path.insert(0, os.path.join(ROOT, 'models'))
sys.path.insert(0, ROOT)
print('ROOT   =', ROOT)
print('MODELS =', MODELS)
print('CHARTS =', CHARTS)

## 1. Feature importance của Random Forest (Tier 2)
`feature_importance.png` — nạp `rf_final.pkl`, lấy `feature_importances_` (Gini), xếp hạng.

In [ ]:
from ai_detector import LogAnomalyDetector

names = LogAnomalyDetector(model_type='if').FEATURE_NAMES
rf = joblib.load(os.path.join(MODELS, 'rf_final.pkl'))
imp = np.asarray(rf.feature_importances_, dtype=float) * 100.0

order = np.argsort(imp)                 # tăng dần -> cao nhất nằm trên cùng
sn = [names[i] for i in order]
si = imp[order]

def colour(v):
    if v >= 10.0: return '#1B7837'      # dominant
    if v >= 1.0:  return '#2E86AB'      # supporting
    return '#B0B0B0'                    # marginal
colours = [colour(v) for v in si]

fig, ax = plt.subplots(figsize=(11, 9))
ypos = np.arange(len(sn))
bars = ax.barh(ypos, si, color=colours)
ax.set_yticks(ypos); ax.set_yticklabels(sn, fontsize=10)
ax.set_xlabel('Gini importance (%)', fontsize=12, fontweight='bold')
ax.set_title('Random Forest (Tier 2) Feature Importance — 22-feature vector',
             fontsize=14, fontweight='bold')
ax.set_xlim(0, max(si) * 1.18)
for b, v in zip(bars, si):
    ax.text(b.get_width() + max(si) * 0.01, b.get_y() + b.get_height() / 2,
            f'{v:.1f}%', va='center', ha='left', fontsize=9)
ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#1B7837', label='Dominant (≥ 10%)'),
    Patch(color='#2E86AB', label='Supporting (1–10%)'),
    Patch(color='#B0B0B0', label='Marginal (< 1%)'),
], loc='lower right', fontsize=10, framealpha=0.9)

plt.tight_layout()
fig.savefig(os.path.join(CHARTS, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. Tier & Hybrid comparison
`model_comparison.png` — đọc `analysis/tier_and_hybrid_results.json`, vẽ 4 metric (Precision/Recall/F1/F2).

In [ ]:
with open(os.path.join(ANALYSIS, 'tier_and_hybrid_results.json')) as f:
    results = json.load(f)

tiers = results['tiers_and_hybrid']
# Rút gọn nhãn: "Smart Consensus (T2 70% + T3 30%)" -> "Smart Consensus"
tier_names = ['Smart Consensus' if t['name'].startswith('Smart Consensus') else t['name']
              for t in tiers]
colors_list = ['#FF6B6B', '#2E86AB', '#06A77D', '#F18F01', '#A23B72']

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('IDS System: Tier & Hybrid Configuration Comparison\n'
             '(Dataset: 111,065 logs, 47.43% attack rate)',
             fontsize=16, fontweight='bold', y=0.995)
x_pos = np.arange(len(tier_names)); width = 0.6

def panel(ax, vals, ylabel, title):
    bars = ax.bar(x_pos, vals, width, color=colors_list)
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks(x_pos); ax.set_xticklabels(tier_names, rotation=30, ha='right', fontsize=10)
    ax.set_ylim(0, 105)
    ax.axhline(y=90, color='green', linestyle='--', alpha=0.3, linewidth=2)
    ax.text(len(tier_names) - 0.5, 92, '90% threshold', fontsize=9, color='green', alpha=0.7)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 1, f'{v:.1f}%',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

panel(axes[0, 0], [t['precision'] for t in tiers], 'Precision (%)',
      'Precision: False Positive Rate (Lower FP = Higher Precision)')
panel(axes[0, 1], [t['recall'] for t in tiers], 'Recall (%)',
      'Recall: Detection Rate (Higher Recall = Fewer Missed Attacks)')
panel(axes[1, 0], [t['f1'] for t in tiers], 'F1-Score (%)',
      'F1-Score: Balance (Precision + Recall)')
panel(axes[1, 1], [t['f2'] for t in tiers], 'F2-Score (%)',
      'F2-Score: IDS-Standard (Recall x 2)')

plt.tight_layout()
fig.savefig(os.path.join(CHARTS, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Comprehensive comparison (4 ảnh)
Đọc `final_model_results.json` (model đơn) + `tier_and_hybrid_results.json` (hybrid) →
`comprehensive_metrics.png`, `precision_vs_recall.png`, `f1_ranking.png`, `radar_comparison.png`.

In [ ]:
import pandas as pd

with open(os.path.join(ANALYSIS, 'final_model_results.json')) as f:
    res = json.load(f)
with open(os.path.join(ANALYSIS, 'tier_and_hybrid_results.json')) as f:
    hyb = {t['name']: t for t in json.load(f)['tiers_and_hybrid']}

data = []
for m in res['supervised']:
    data.append({'Tier': 'Tier 2', 'Model': m['model'], 'Precision': m['precision'],
                 'Recall': m['recall'], 'F1': m['f1'], 'F2': m['f2']})
for m in res['unsupervised']:
    data.append({'Tier': 'Tier 3', 'Model': m['model'], 'Precision': m['precision'],
                 'Recall': m['recall'], 'F1': m['f1'], 'F2': m['f2']})
s = hyb['Smart Consensus (T2 70% + T3 30%)']
data.append({'Tier': 'Hybrid', 'Model': 'Smart Consensus', 'Precision': s['precision'],
             'Recall': s['recall'], 'F1': s['f1'], 'F2': s['f2']})
v = hyb['Simple Voting (T1 OR T2 OR T3)']
data.append({'Tier': 'Hybrid', 'Model': 'Simple Voting', 'Precision': v['precision'],
             'Recall': v['recall'], 'F1': v['f1'], 'F2': v['f2']})
df = pd.DataFrame(data)

tier_colors = {'Tier 2': '#2ecc71', 'Tier 3': '#3498db', 'Hybrid': '#e74c3c'}
colors = [tier_colors[t] for t in df['Tier']]

# --- Chart 1: comprehensive_metrics ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Comprehensive Model Performance Comparison', fontsize=16, fontweight='bold', y=1.00)
def barpanel(ax, col, title):
    bars = ax.bar(range(len(df)), df[col], color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.set_ylabel(col + ' (%)', fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(df))); ax.set_xticklabels(df['Model'], rotation=45, ha='right', fontsize=9)
    ax.set_ylim(0, 105); ax.grid(axis='y', alpha=0.3, linestyle='--')
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f'{b.get_height():.1f}%',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
barpanel(axes[0, 0], 'Precision', 'Precision (True Positive Rate)')
barpanel(axes[0, 1], 'Recall', 'Recall (Detection Rate)')
barpanel(axes[1, 0], 'F1', 'F1-Score (Harmonic Mean)')
barpanel(axes[1, 1], 'F2', 'F2-Score (IDS-Standard: Recall x 2)')
plt.tight_layout()
fig.savefig(os.path.join(CHARTS, 'comprehensive_metrics.png'), dpi=300, bbox_inches='tight')
plt.show()

# --- Chart 2: precision_vs_recall ---
fig, ax = plt.subplots(figsize=(12, 8))
for tier in ['Tier 2', 'Tier 3', 'Hybrid']:
    d = df[df['Tier'] == tier]
    ax.scatter(d['Recall'], d['Precision'], s=300, alpha=0.7, color=tier_colors[tier],
               edgecolor='black', linewidth=2, label=tier)
for _, r in df.iterrows():
    ax.annotate(r['Model'], (r['Recall'], r['Precision']), xytext=(5, 5),
                textcoords='offset points', fontsize=9, fontweight='bold')
ax.set_xlabel('Recall (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Precision (%)', fontsize=12, fontweight='bold')
ax.set_title('Precision vs Recall Trade-off', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--'); ax.legend(fontsize=11, loc='best')
ax.set_xlim(0, 105); ax.set_ylim(0, 105)
plt.tight_layout()
fig.savefig(os.path.join(CHARTS, 'precision_vs_recall.png'), dpi=300, bbox_inches='tight')
plt.show()

# --- Chart 3: f1_ranking ---
fig, ax = plt.subplots(figsize=(12, 8))
ds = df.sort_values('F1', ascending=True)
cs = [tier_colors[t] for t in ds['Tier']]
bars = ax.barh(range(len(ds)), ds['F1'], color=cs, alpha=0.8, edgecolor='black', linewidth=2)
ax.set_yticks(range(len(ds))); ax.set_yticklabels(ds['Model'], fontsize=11, fontweight='bold')
ax.set_xlabel('F1-Score (%)', fontsize=12, fontweight='bold')
ax.set_title('Model Ranking by F1-Score', fontsize=14, fontweight='bold')
ax.set_xlim(0, 105); ax.grid(axis='x', alpha=0.3, linestyle='--')
for i, (b, val) in enumerate(zip(bars, ds['F1'])):
    ax.text(val + 1, i, f'{val:.2f}%', va='center', fontweight='bold', fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(CHARTS, 'f1_ranking.png'), dpi=300, bbox_inches='tight')
plt.show()

# --- Chart 4: radar_comparison ---
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
cats = ['Precision', 'Recall', 'F1', 'F2']; N = len(cats)
angles = [n / float(N) * 2 * np.pi for n in range(N)]; angles += angles[:1]
rf_d = df[df['Model'] == 'RandomForest'].iloc[0]
lof_d = df[df['Model'] == 'Local Outlier Factor'].iloc[0]
hb_d = df[df['Model'] == 'Smart Consensus'].iloc[0]
for d, c, lab in [(rf_d, '#2ecc71', 'RandomForest (Tier 2)'),
                  (lof_d, '#3498db', 'LOF (Tier 3)'),
                  (hb_d, '#e74c3c', 'Smart Consensus')]:
    vals = [d['Precision'], d['Recall'], d['F1'], d['F2']]; vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=lab, color=c, markersize=8)
    ax.fill(angles, vals, alpha=0.25, color=c)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(cats, fontsize=11, fontweight='bold')
ax.set_ylim(0, 105); ax.set_yticks([20, 40, 60, 80, 100])
ax.set_yticklabels(['20%', '40%', '60%', '80%', '100%'], fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(fontsize=11, loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.set_title('Multi-Metric Performance Comparison', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
fig.savefig(os.path.join(CHARTS, 'radar_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

## 4. Confusion matrices
Nạp 6 model `*_final.pkl` và **chạy inference** (không train lại) trên
`datasets/final_dataset_eval.log` → dựng ma trận nhầm lẫn cho 8 cấu hình.
> Bước này cần file `final_dataset_eval.log`. Chạy vài giây đến ~1 phút.

In [ ]:
import re
from urllib.parse import urlparse, unquote, parse_qs
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, fbeta_score)
import apache_log
apache_log.PATH_PARAM_VOCAB = {}   # model cuối train với vocab rỗng

EVAL_FILE = os.path.join(ROOT, 'datasets', 'final_dataset_eval.log')
LOG_RE = re.compile(
    r'(\S+) - - \[([^\]]+)\] "(\S+)\s+([^"]+)(?:\s+HTTP[^"]*)?"\s+(\d+)\s+(\S+)\s+"([^"]*)"\s+"([^"]*)"')

det = LogAnomalyDetector()
X, y, regex_hits = [], [], []
with open(EVAL_FILE, errors='ignore') as f:
    for line in f:
        line = line.strip()
        m = LOG_RE.match(line)
        if not m:
            continue
        ip, ts, method, url, status, size, ref, ua = m.groups()
        try:
            durl = unquote(url)
        except Exception:
            durl = url
        pu = urlparse(durl); path = pu.path or '/'; query = pu.query or ''
        try:
            feats = det.extract_features(path, query, method, str(status), ua, ref)
        except Exception:
            continue
        X.append(feats); y.append(1 if '(Simulated-Attack)' in line else 0)
        pdd = {'path': path, 'query': query,
               'params': parse_qs(query, keep_blank_values=True),
               'user_agent': ua, 'method': method}
        regex_hits.append(1 if len(apache_log.detect_rule_based(pdd)) > 0 else 0)

X = np.array(X); y = np.array(y); regex_hits = np.array(regex_hits)
n = len(y); n_atk = int(y.sum())
print(f'eval = {n} samples ({n_atk} attack, {n - n_atk} clean, {n_atk / n * 100:.2f}% attack)')

rf = joblib.load(os.path.join(MODELS, 'rf_final.pkl'))
lr = joblib.load(os.path.join(MODELS, 'lr_final.pkl'))
iso = joblib.load(os.path.join(MODELS, 'isolation_forest_final.pkl'))
ocsvm = joblib.load(os.path.join(MODELS, 'ocsvm_final.pkl'))
lof = joblib.load(os.path.join(MODELS, 'lof_final.pkl'))
scaler = joblib.load(os.path.join(MODELS, 'scaler_final.pkl'))

Xs = scaler.transform(X)
rf_pred = rf.predict(X); rf_proba = rf.predict_proba(X)[:, 1]
lr_pred = lr.predict(X)
if_pred = (iso.predict(X) == -1).astype(int)
ocsvm_pred = (ocsvm.predict(Xs) == -1).astype(int)
lof_pred = (lof.predict(Xs) == -1).astype(int)

smart = ((regex_hits == 1) | (rf_proba >= 0.5) | ((lof_pred == 1) & (rf_proba >= 0.3))).astype(int)
voting = ((regex_hits == 1) | (rf_proba >= 0.5) | (lof_pred == 1)).astype(int)

panels = [
    ('Tier 1: Regex Rules', regex_hits),
    ('Tier 2: RandomForest', rf_pred),
    ('Tier 2: LogisticRegression', lr_pred),
    ('Tier 3: Isolation Forest', if_pred),
    ('Tier 3: One-Class SVM', ocsvm_pred),
    ('Tier 3: Local Outlier Factor', lof_pred),
    ('Smart Consensus', smart),
    ('Simple Voting', voting),
]

cols = 4; rows = (len(panels) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.4 * rows)); axes = axes.flatten()
for i, (name, pred) in enumerate(panels):
    cm = confusion_matrix(y, pred, labels=[0, 1])
    f1 = f1_score(y, pred, zero_division=0) * 100
    f2 = fbeta_score(y, pred, beta=2, zero_division=0) * 100
    ax = axes[i]; ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred Clean', 'Pred Attack'], fontsize=9)
    ax.set_yticklabels(['True Clean', 'True Attack'], fontsize=9)
    ax.set_title(f'{name}\nF1={f1:.1f}  F2={f2:.1f}', fontsize=10)
    vmax = cm.max()
    for rr in range(2):
        for cc in range(2):
            ax.text(cc, rr, f'{cm[rr, cc]}', ha='center', va='center',
                    color=('white' if cm[rr, cc] > vmax * 0.5 else 'black'),
                    fontsize=12, fontweight='bold')
for j in range(len(panels), len(axes)):
    axes[j].axis('off')
fig.suptitle(f'Confusion Matrices on the Evaluation Set ({n:,} logs, {n_atk / n * 100:.2f}% attack)',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(os.path.join(CHARTS, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()